# 06 - Fusion Results Review

This notebook compares the training outputs from:

1. `04_loso_tabular_baselines.ipynb`
2. `05_loso_two_tower_fusion.ipynb`

It produces a concise model-comparison report for thesis writing.

In [ ]:
# SECTION 1: Imports + Paths
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass
    ROOT = Path('/content/drive/MyDrive')
else:
    ROOT = Path.home() / 'Desktop' / 'thesis'

TAB_DIR = ROOT / 'research_outputs' / 'fusion_training' / 'v1_loso_tabular'
NN_DIR = ROOT / 'research_outputs' / 'fusion_training' / 'v1_loso_two_tower'
OUT_DIR = ROOT / 'research_outputs' / 'fusion_training' / 'v1_review'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Paths set')
print('TAB_DIR:', TAB_DIR)
print('NN_DIR :', NN_DIR)
print('OUT_DIR:', OUT_DIR)

In [ ]:
# SECTION 2: Load Results
tab_fold = TAB_DIR / 'loso_fold_metrics.csv'
tab_agg = TAB_DIR / 'loso_aggregate_metrics.csv'
nn_fold = NN_DIR / 'loso_fold_metrics.csv'
nn_agg = NN_DIR / 'loso_aggregate_metrics.json'

if not tab_fold.exists() or not tab_agg.exists():
    raise FileNotFoundError('Tabular baseline outputs not found. Run notebook 04 first.')
if not nn_fold.exists() or not nn_agg.exists():
    raise FileNotFoundError('Two-tower outputs not found. Run notebook 05 first.')

tab_fold_df = pd.read_csv(tab_fold)
tab_agg_df = pd.read_csv(tab_agg)
nn_fold_df = pd.read_csv(nn_fold)
with open(nn_agg, 'r') as f:
    nn_agg_obj = json.load(f)

nn_agg_df = pd.DataFrame([
    {
        'model': 'two_tower',
        'macro_f1_mean': nn_agg_obj['macro_f1_mean'],
        'macro_f1_std': nn_agg_obj['macro_f1_std'],
        'balanced_acc_mean': nn_agg_obj['balanced_acc_mean'],
        'accuracy_mean': nn_agg_obj['accuracy_mean'],
        'folds': nn_agg_obj['folds']
    }
])

print('✓ Results loaded')
display(tab_agg_df)
display(nn_agg_df)

In [ ]:
# SECTION 3: Build Unified Comparison
cmp_df = pd.concat([tab_agg_df, nn_agg_df], ignore_index=True, sort=False)
cmp_df = cmp_df.sort_values('macro_f1_mean', ascending=False).reset_index(drop=True)

print('Unified comparison:')
display(cmp_df)

best_model = cmp_df.iloc[0]['model']
print('Best by macro_f1_mean:', best_model)

In [ ]:
# SECTION 4: Plot and Export
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cmp_df.plot(x='model', y='macro_f1_mean', kind='bar', legend=False, ax=axes[0], color='tab:blue')
axes[0].set_title('Macro-F1 Mean by Model')
axes[0].set_ylabel('Macro-F1')
axes[0].tick_params(axis='x', rotation=45)

cmp_df.plot(x='model', y='balanced_acc_mean', kind='bar', legend=False, ax=axes[1], color='tab:green')
axes[1].set_title('Balanced Accuracy Mean by Model')
axes[1].set_ylabel('Balanced Acc')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plot_path = OUT_DIR / 'model_comparison.png'
plt.savefig(plot_path, dpi=150)
plt.show()

cmp_path = OUT_DIR / 'model_comparison.csv'
cmp_df.to_csv(cmp_path, index=False)

summary = {
    'best_model_by_macro_f1': str(best_model),
    'models_evaluated': cmp_df['model'].astype(str).tolist(),
    'comparison_csv': str(cmp_path),
    'comparison_plot': str(plot_path)
}
summary_path = OUT_DIR / 'review_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('✓ Review exports created')
print('  -', cmp_path)
print('  -', plot_path)
print('  -', summary_path)
print('Summary:', summary)